# 02 — RFM & Cohort Analysis

**Goal:** Build the full RFM feature set and cohort retention matrix that will feed the BG/NBD model in Phase 2.

**Tools:** Polars (lazy pipeline), DuckDB (SQL aggregations), Plotly (charts)  
**Outputs:** `rfm_features` and `customers` rows in Supabase.

---

### Sections
1. Load and split data (calibration / holdout)
2. Compute RFM features with Polars
3. Compute cohort features
4. Build ground-truth LTV labels from holdout
5. Cohort retention matrix
6. LTV development over time by cohort
7. W&B run tracking
8. Save to Supabase


In [1]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import polars as pl
from IPython.display import display

sys.path.insert(0, str(Path().resolve().parent))

from backend.config import settings
from backend.data.load_data import load_uci_csv
from backend.features.rfm import (
    clean_transactions,
    assign_product_categories,
    assign_amount_buckets,
    make_calibration_holdout_split,
    RFMPipeline,
)
from backend.features.cohorts import CohortPipeline
from backend.features.duckdb_agg import DuckDBAggregator
from backend.db.supabase_client import SupabaseClient

import plotly.io as pio
pio.templates.default = 'plotly_white'

print('✓ Imports OK')

✓ Imports OK


In [2]:
# ── Config ────────────────────────────────────────────────────────────
OBSERVATION_MONTHS = 6
HOLDOUT_MONTHS     = 6
SAVE_TO_DB         = True   # set False to skip Supabase writes
USE_WANDB          = True   # set False to skip W&B logging

print(f'Observation window: {OBSERVATION_MONTHS} months')
print(f'Holdout window:     {HOLDOUT_MONTHS} months')

Observation window: 6 months
Holdout window:     6 months


## 1. Load and Split Data

In [3]:
# Load raw and clean
raw_df  = load_uci_csv(settings.UCI_CSV_PATH)
cleaned = clean_transactions(raw_df)
cleaned = assign_product_categories(cleaned)
cleaned = assign_amount_buckets(cleaned)

print(f'Cleaned transactions: {len(cleaned):,}')

# Calibration / holdout split
calibration, holdout, obs_end, holdout_end = make_calibration_holdout_split(
    cleaned,
    observation_months=OBSERVATION_MONTHS,
    holdout_months=HOLDOUT_MONTHS,
)

print(f'\nCalibration: {len(calibration):,} rows  (up to {obs_end})')
print(f'Holdout:     {len(holdout):,} rows  ({obs_end} – {holdout_end})')
print(f'\nCalibration customers: {calibration["customer_id"].n_unique():,}')
print(f'Holdout customers:     {holdout["customer_id"].n_unique():,}')

2026-04-21 12:47:45.051 | INFO     | backend.data.load_data:load_uci_csv:75 - Loading UCI dataset from: E:\DOWNLOADS\DATA SCIENCE\PROFESSIONAL PROJECTS\CUSTOMER_LIFE_TIME_VALUE\backend\data\raw\OnlineRetail.csv
2026-04-21 12:47:45.052 | INFO     | backend.data.load_data:load_uci_csv:93 - Reading CSV with DuckDB for robust date parsing...
2026-04-21 12:47:48.911 | INFO     | backend.data.load_data:load_uci_csv:130 - Loaded 541909 rows — 8 columns — date range 2010-12-01 08:26:00+00:00 to 2011-12-09 12:50:00+00:00
2026-04-21 12:47:48.913 | INFO     | backend.features.rfm:clean_transactions:54 - Cleaning transactions — 541909 raw rows
2026-04-21 12:47:48.929 | INFO     | backend.features.rfm:clean_transactions:76 - After cleaning: 397884 rows (73.4% kept)
2026-04-21 12:47:48.951 | INFO     | backend.features.rfm:make_calibration_holdout_split:157 - Split → calibration 144541 rows (≤2011-05-30), holdout 225975 rows (2011-05-30 – 2011-11-26)


Cleaned transactions: 397,884

Calibration: 144,541 rows  (up to 2011-05-30)
Holdout:     225,975 rows  (2011-05-30 – 2011-11-26)

Calibration customers: 2,708
Holdout customers:     3,429


In [4]:
# Quick split sanity check
fig = make_subplots(rows=1, cols=2, subplot_titles=['Calibration', 'Holdout'])

for i, (df, label) in enumerate([(calibration, 'Calibration'), (holdout, 'Holdout')], start=1):
    monthly = (
        df.with_columns(pl.col('invoice_date').dt.strftime('%Y-%m').alias('month'))
          .group_by('month').agg((pl.col('quantity') * pl.col('unit_price')).sum().alias('rev'))
          .sort('month')
    )
    fig.add_trace(go.Bar(x=monthly['month'].to_list(), y=monthly['rev'].to_list(), name=label), row=1, col=i)

fig.update_layout(height=350, title='Monthly Revenue: Calibration vs Holdout', showlegend=False)
fig.show()

## 2. Compute RFM Features

In [5]:
# Run the RFM pipeline on calibration data
rfm_pipeline = RFMPipeline(
    transactions=calibration,
    observation_end_date=obs_end,
)

rfm_df = rfm_pipeline.compute()
print(f'RFM computed for {len(rfm_df):,} customers')
print(f'Columns: {rfm_df.columns}')

2026-04-21 12:47:50.030 | INFO     | backend.features.rfm:compute:191 - Computing RFM for 144541 transactions up to 2011-05-30
2026-04-21 12:47:50.066 | INFO     | backend.features.rfm:compute:283 - RFM computed for 2708 customers — obs_end=2011-05-30


RFM computed for 2,708 customers
Columns: ['customer_id', 'frequency', 'monetary_avg', 'monetary_std', 'monetary_total', 'purchase_variance', 'first_purchase_date', 'last_purchase_date', 'unique_invoices', 'avg_items_per_order', 'first_purchase_category', 'first_purchase_amount', 'unique_categories', 'multi_country', 'recency_days', 't_days', 'cohort_month', 'observation_end_date', 'pipeline_run_id', '_n_source_rows', 'orders_count', 'avg_days_between_orders', 'std_days_between_orders', 'days_to_second_purchase', 'unique_products']


In [6]:
# Preview
display(rfm_df.select([
    'customer_id', 'frequency', 'recency_days', 't_days',
    'monetary_avg', 'monetary_total', 'orders_count',
    'cohort_month', 'days_to_second_purchase'
]).head(10))

customer_id,frequency,recency_days,t_days,monetary_avg,monetary_total,orders_count,cohort_month,days_to_second_purchase
str,i32,f64,i32,f64,f64,u32,str,i32
"""18287""",0,0.0,8,765.28,765.28,1,"""2011-05""",null
"""16057""",1,73.0,91,353.18,1059.54,3,"""2011-02""",73
"""17711""",0,0.0,96,220.53,220.53,1,"""2011-02""",null
"""18048""",0,0.0,10,86.145,172.29,2,"""2011-05""",null
"""14701""",4,69.0,123,350.15,1750.75,5,"""2011-01""",19
"""12501""",0,0.0,143,2169.39,2169.39,1,"""2011-01""",null
"""14907""",1,7.0,174,320.23,960.69,3,"""2010-12""",7
"""16103""",1,7.0,40,267.23,534.46,2,"""2011-04""",7
"""15434""",0,0.0,7,306.96,306.96,1,"""2011-05""",null


In [7]:
# Summary statistics — the ones BG/NBD needs most
bgnbd_cols = ['frequency', 'recency_days', 't_days', 'monetary_avg']
print('=== BG/NBD Input Statistics ===')
display(rfm_df.select(bgnbd_cols).describe())

# How many customers have frequency > 0 (repeat buyers)?
repeat = (rfm_df['frequency'] > 0).sum()
print(f'\nRepeat buyers (freq>0): {repeat:,} / {len(rfm_df):,}  ({100*repeat/len(rfm_df):.1f}%)')

=== BG/NBD Input Statistics ===


statistic,frequency,recency_days,t_days,monetary_avg
str,f64,f64,f64,f64
"""count""",2708.0,2708.0,2708.0,2708.0
"""null_count""",0.0,0.0,0.0,0.0
"""mean""",1.464549,48.117061,110.839734,408.69922
"""std""",3.139509,60.070516,54.440291,1558.124707
"""min""",0.0,0.0,1.0,2.9
"""25%""",0.0,0.0,67.0,172.77
"""50%""",0.0,0.0,117.0,291.946667
"""75%""",2.0,99.0,168.0,423.33
"""max""",46.0,179.0,180.0,77183.6



Repeat buyers (freq>0): 1,327 / 2,708  (49.0%)


In [8]:
# Calibration check: frequency vs recency (BG/NBD standard diagnostic)
# Customers with higher frequency should generally have higher recency
fig = px.scatter(
    rfm_df.filter(pl.col('frequency') <= 30).to_pandas(),
    x='t_days', y='recency_days',
    color='frequency',
    color_continuous_scale='Viridis',
    opacity=0.5,
    title='T vs Recency (coloured by Frequency) — BG/NBD Calibration Diagnostic',
    labels={'t_days': 'Customer Age T (days)', 'recency_days': 'Recency (days)'},
)
# Add T = recency line (upper bound)
max_t = rfm_df['t_days'].max()
fig.add_trace(go.Scatter(x=[0, max_t], y=[0, max_t], mode='lines',
                          line=dict(color='red', dash='dash'), name='T = recency'))
fig.show()

In [9]:
# Days to second purchase — strong churn predictor
d2s = rfm_df.filter(pl.col('days_to_second_purchase').is_not_null())
d2s_capped = d2s.filter(pl.col('days_to_second_purchase') <= 120)

fig = px.histogram(
    d2s_capped.to_pandas(),
    x='days_to_second_purchase', nbins=60,
    title='Days to Second Purchase (≤120 days)',
    labels={'days_to_second_purchase': 'Days to 2nd Purchase'},
)
# Add 30-day vertical line
fig.add_vline(x=30, line_dash='dash', line_color='red',
              annotation_text='30 days', annotation_position='top right')
fig.show()

fast_return = (d2s['days_to_second_purchase'] <= 30).sum()
print(f'Returned within 30 days: {fast_return:,} / {len(d2s):,} ({100*fast_return/len(d2s):.1f}%)')

Returned within 30 days: 515 / 1,327 (38.8%)


## 3. Cohort Features

In [10]:
cohort_pipeline = CohortPipeline(calibration)

# Cohort assignments
cohort_df = cohort_pipeline.compute_cohort_assignments()
cohort_sizes = cohort_pipeline.compute_cohort_sizes(cohort_df)

print(f'{len(cohort_df)} customers across {cohort_df["cohort_month"].n_unique()} monthly cohorts')
display(cohort_sizes)

2026-04-21 12:47:56.049 | INFO     | backend.features.cohorts:compute_cohort_assignments:73 - Cohort assignments — 2708 customers across 6 cohorts


2708 customers across 6 monthly cohorts


cohort_month,cohort_size
str,u32
"""2010-12""",885
"""2011-01""",417
"""2011-02""",380
"""2011-03""",452
"""2011-04""",300
"""2011-05""",274


In [11]:
# Cohort sizes over time
fig = px.bar(
    cohort_sizes.to_pandas(),
    x='cohort_month', y='cohort_size',
    title='Monthly Cohort Acquisition Size',
    labels={'cohort_month': 'Cohort Month', 'cohort_size': 'New Customers'},
    color='cohort_size', color_continuous_scale='Blues',
)
fig.show()

In [12]:
# Enrich RFM with cohort metadata
rfm_enriched = cohort_pipeline.enrich_rfm(rfm_df, cohort_df, obs_end)
print(f'Enriched RFM columns: {rfm_enriched.columns}')

Enriched RFM columns: ['customer_id', 'frequency', 'monetary_avg', 'monetary_std', 'monetary_total', 'purchase_variance', 'first_purchase_date', 'last_purchase_date', 'unique_invoices', 'avg_items_per_order', 'first_purchase_category', 'first_purchase_amount', 'unique_categories', 'multi_country', 'recency_days', 't_days', 'cohort_month', 'observation_end_date', 'pipeline_run_id', '_n_source_rows', 'orders_count', 'avg_days_between_orders', 'std_days_between_orders', 'days_to_second_purchase', 'unique_products', 'cohort_start_date', 'customer_age_days']


## 4. Ground-Truth LTV Labels

In [13]:
# Attach 12m LTV labels from holdout
rfm_labelled = rfm_pipeline.compute_ltv_labels(
    holdout_df=holdout,
    rfm_df=rfm_enriched,
    horizon_months=12,
)

n_nonzero = (rfm_labelled['actual_ltv_12m'] > 0).sum()
print(f'Customers with >0 LTV in holdout: {n_nonzero:,} / {len(rfm_labelled):,}  ({100*n_nonzero/len(rfm_labelled):.1f}%)')
print(f'\nActual 12m LTV distribution:')
print(rfm_labelled.filter(pl.col('actual_ltv_12m') > 0)['actual_ltv_12m'].describe())

2026-04-21 12:47:59.867 | INFO     | backend.features.rfm:compute_ltv_labels:376 - Computing actual LTV labels for 12m horizon (225975 holdout rows)
2026-04-21 12:47:59.879 | INFO     | backend.features.rfm:compute_ltv_labels:401 - actual_ltv_12m label — 1875 / 2708 customers have >0 holdout revenue


Customers with >0 LTV in holdout: 1,875 / 2,708  (69.2%)

Actual 12m LTV distribution:
shape: (9, 2)
┌────────────┬─────────────┐
│ statistic  ┆ value       │
│ ---        ┆ ---         │
│ str        ┆ f64         │
╞════════════╪═════════════╡
│ count      ┆ 1875.0      │
│ null_count ┆ 0.0         │
│ mean       ┆ 2031.247638 │
│ std        ┆ 7844.110037 │
│ min        ┆ 7.5         │
│ 25%        ┆ 356.99      │
│ 50%        ┆ 785.98      │
│ 75%        ┆ 1674.07     │
│ max        ┆ 183243.57   │
└────────────┴─────────────┘


In [14]:
# LTV distribution
ltv_pos = rfm_labelled.filter(pl.col('actual_ltv_12m') > 0)
p99 = ltv_pos['actual_ltv_12m'].quantile(0.99)

fig = px.histogram(
    ltv_pos.filter(pl.col('actual_ltv_12m') <= p99).to_pandas(),
    x='actual_ltv_12m', nbins=80,
    title=f'Actual 12m LTV Distribution (positive buyers, ≤ £{p99:.0f} p99)',
    labels={'actual_ltv_12m': 'Actual 12m LTV (£)'},
)
fig.show()

# How skewed?
mean_ltv   = ltv_pos['actual_ltv_12m'].mean()
median_ltv = ltv_pos['actual_ltv_12m'].median()
print(f'Mean LTV:   £{mean_ltv:.2f}')
print(f'Median LTV: £{median_ltv:.2f}')
print(f'Skew ratio (mean/median): {mean_ltv/median_ltv:.2f}x  — highly right-skewed, Huber loss needed')

Mean LTV:   £2031.25
Median LTV: £785.98
Skew ratio (mean/median): 2.58x  — highly right-skewed, Huber loss needed


In [15]:
# Frequency vs actual LTV — key validation
filtered = rfm_labelled.filter(
    (pl.col('actual_ltv_12m') > 0) &
    (pl.col('actual_ltv_12m') < rfm_labelled['actual_ltv_12m'].quantile(0.98))
)
sample = filtered.sample(min(2000, len(filtered)))

fig = px.scatter(
    sample.to_pandas(),
    x='frequency', y='actual_ltv_12m',
    color='monetary_avg',
    color_continuous_scale='RdYlGn',
    opacity=0.5,
    trendline='ols',
    title='Frequency vs Actual 12m LTV (coloured by avg order value)',
    labels={'frequency': 'Calibration Frequency', 'actual_ltv_12m': 'Actual 12m LTV (£)'},
)
fig.show()


## 5. Cohort Retention Matrix

In [16]:
# Full retention matrix (calibration + holdout combined)
cohort_pipeline_full = CohortPipeline(cleaned)  # use all data
cohort_df_full = cohort_pipeline_full.compute_cohort_assignments()
retention = cohort_pipeline_full.compute_retention_matrix(cohort_df_full, max_months=11)

print(f'Retention matrix: {len(retention)} rows')
retention.head(10)

2026-04-21 12:48:02.978 | INFO     | backend.features.cohorts:compute_cohort_assignments:73 - Cohort assignments — 4338 customers across 13 cohorts
2026-04-21 12:48:03.002 | INFO     | backend.features.cohorts:compute_retention_matrix:164 - Retention matrix — 13 cohorts × 11 months


Retention matrix: 80 rows


cohort_month,months_since_first,active_customers,cohort_n,retention_rate_pct
str,i32,u32,u32,f64
"""2010-12""",0,885,885,100.0
"""2010-12""",1,324,885,36.61
"""2010-12""",2,452,885,51.07
"""2010-12""",3,321,885,36.27
"""2010-12""",4,352,885,39.77
"""2010-12""",5,321,885,36.27
"""2010-12""",6,309,885,34.92
"""2010-12""",7,313,885,35.37
"""2010-12""",9,480,885,54.24


In [17]:
# Pivot to wide format for heatmap
retention_pivot = retention.pivot(
    values='retention_rate_pct',
    index='cohort_month',
    on='months_since_first',
    aggregate_function='first',
).sort('cohort_month')

cohorts = retention_pivot['cohort_month'].to_list()
month_cols = [c for c in retention_pivot.columns if c != 'cohort_month']
z = retention_pivot.select(month_cols).to_numpy()

fig = go.Figure(go.Heatmap(
    z=z,
    x=[f'M+{m}' for m in month_cols],
    y=cohorts,
    colorscale='RdYlGn',
    zmin=0, zmax=100,
    text=[[f'{v:.0f}%' if v is not None else '' for v in row] for row in z.tolist()],
    texttemplate='%{text}',
    textfont=dict(size=10),
    colorbar=dict(title='Retention %'),
))
fig.update_layout(
    title='Cohort Retention Matrix',
    xaxis_title='Months Since First Purchase',
    yaxis_title='Acquisition Cohort',
    height=450,
)
fig.show()

## 6. LTV Development Over Time by Cohort

In [18]:
ltv_curves = cohort_pipeline_full.compute_cohort_ltv_over_time(
    cohort_df_full, max_months=11
)
print(f'LTV curve rows: {len(ltv_curves)}')

fig = px.line(
    ltv_curves.to_pandas(),
    x='months_since_first',
    y='cumulative_ltv_per_customer',
    color='cohort_month',
    title='Cumulative LTV per Customer by Cohort',
    labels={
        'months_since_first': 'Months Since First Purchase',
        'cumulative_ltv_per_customer': 'Cumulative LTV / Customer (£)',
    },
)
fig.show()

LTV curve rows: 80


## 7. W&B Logging

In [19]:
if USE_WANDB:
    try:
        import wandb
        
        run = wandb.init(
            project=settings.WANDB_PROJECT,
            name='week1_rfm_cohort',
            tags=['week1', 'rfm', 'feature_engineering'],
            config={
                'observation_months':    OBSERVATION_MONTHS,
                'holdout_months':        HOLDOUT_MONTHS,
                'observation_end':       str(obs_end),
                'holdout_end':           str(holdout_end),
                'n_raw_rows':            len(raw_df),
                'n_cleaned_rows':        len(cleaned),
                'n_calibration_rows':    len(calibration),
                'n_holdout_rows':        len(holdout),
                'n_customers':           len(rfm_labelled),
                'n_repeat_buyers':       int((rfm_labelled['frequency'] > 0).sum()),
                'pct_nonzero_ltv_12m':   round(100 * n_nonzero / len(rfm_labelled), 2),
            }
        )
        
        # Log summary stats
        wandb.log({
            'mean_ltv_12m':           float(rfm_labelled.filter(pl.col('actual_ltv_12m') > 0)['actual_ltv_12m'].mean()),
            'median_ltv_12m':         float(rfm_labelled.filter(pl.col('actual_ltv_12m') > 0)['actual_ltv_12m'].median()),
            'mean_frequency':         float(rfm_labelled['frequency'].mean()),
            'mean_monetary_avg':      float(rfm_labelled['monetary_avg'].mean()),
            'n_customers':            len(rfm_labelled),
            'n_cohorts':              rfm_labelled['cohort_month'].n_unique(),
        })
        
        # Log RFM table
        rfm_sample = rfm_labelled.sample(min(500, len(rfm_labelled)))
        wandb.log({'rfm_sample': wandb.Table(dataframe=rfm_sample.to_pandas())})
        
        wandb.finish()
        print('✓ W&B run logged')
    except Exception as e:
        print(f'W&B logging skipped: {e}')
else:
    print('W&B logging disabled (USE_WANDB=False)')

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: rakshitkaintura2005 (rakshitkaintura2005-sir-m-visvesvaraya-institute-of-tech) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


mean_frequency,▁
mean_ltv_12m,▁
mean_monetary_avg,▁
median_ltv_12m,▁
n_cohorts,▁
n_customers,▁
mean_frequency,1.46455
mean_ltv_12m,2031.24764
mean_monetary_avg,408.69922
median_ltv_12m,785.98
n_cohorts,6


✓ W&B run logged


## 8. Save to Supabase

In [20]:
if SAVE_TO_DB:
    db = SupabaseClient(use_service_role=True)
    assert db.health_check(), 'DB health check failed — check .env'
    
    print('Saving RFM features to Supabase...')
    n_saved = rfm_pipeline.save(rfm_labelled, db)
    print(f'✓ Saved {n_saved:,} RFM rows')
else:
    print('Skipping DB save (SAVE_TO_DB=False)')
    print(f'\nRFM DataFrame ready with {len(rfm_labelled):,} rows')

2026-04-21 12:48:24.176 | INFO     | backend.db.supabase_client:get_supabase_admin_client:47 - Supabase admin client initialised (service-role)
2026-04-21 12:48:24.346 | INFO     | backend.db.supabase_client:get_db_engine:77 - SQLAlchemy sync engine created — pool_size=5, max_overflow=10
2026-04-21 12:48:25.585 | INFO     | backend.features.rfm:save:416 - Saving 2708 RFM rows to Supabase…


Saving RFM features to Supabase...


2026-04-21 12:48:50.870 | DEBUG    | backend.db.supabase_client:bulk_upsert:219 - Upserted batch 1/6 into rfm_features (500 rows)
2026-04-21 12:49:15.501 | DEBUG    | backend.db.supabase_client:bulk_upsert:219 - Upserted batch 2/6 into rfm_features (500 rows)
2026-04-21 12:49:43.579 | DEBUG    | backend.db.supabase_client:bulk_upsert:219 - Upserted batch 3/6 into rfm_features (500 rows)
2026-04-21 12:50:09.905 | DEBUG    | backend.db.supabase_client:bulk_upsert:219 - Upserted batch 4/6 into rfm_features (500 rows)
2026-04-21 12:50:36.131 | DEBUG    | backend.db.supabase_client:bulk_upsert:219 - Upserted batch 5/6 into rfm_features (500 rows)
2026-04-21 12:50:46.827 | DEBUG    | backend.db.supabase_client:bulk_upsert:219 - Upserted batch 6/6 into rfm_features (208 rows)
2026-04-21 12:50:46.867 | INFO     | backend.db.supabase_client:bulk_upsert:227 - bulk_upsert → 2708 rows into rfm_features
2026-04-21 12:50:46.868 | INFO     | backend.features.rfm:save:438 - Saved 2708 RFM rows


✓ Saved 2,708 RFM rows


In [21]:
# Final summary
print('\n=== Week 1 Complete — Summary ===')
print(f'  Calibration customers:     {len(rfm_labelled):,}')
print(f'  Observation window:        up to {obs_end}')
print(f'  Holdout window:            {obs_end} – {holdout_end}')
print(f'  Repeat buyers (freq > 0):  {int((rfm_labelled["frequency"] > 0).sum()):,}')
print(f'  Holdout LTV coverage:      {100*n_nonzero/len(rfm_labelled):.1f}% have >0 actual LTV')
print(f'  Mean actual 12m LTV:       £{rfm_labelled.filter(pl.col("actual_ltv_12m")>0)["actual_ltv_12m"].mean():.2f}')
print(f'  Cohort retention saved:    {len(retention)} matrix rows')
print(f'\nNext: notebooks/03_bgnbd_gamma_gamma.ipynb (Phase 2)')


=== Week 1 Complete — Summary ===
  Calibration customers:     2,708
  Observation window:        up to 2011-05-30
  Holdout window:            2011-05-30 – 2011-11-26
  Repeat buyers (freq > 0):  1,327
  Holdout LTV coverage:      69.2% have >0 actual LTV
  Mean actual 12m LTV:       £2031.25
  Cohort retention saved:    80 matrix rows

Next: notebooks/03_bgnbd_gamma_gamma.ipynb (Phase 2)
